# 0x8dxd 按市场聚合：交易行为、盈利、回撤、出手次数

**数据源**：
- `trades_cleaned.csv`：成交明细（买卖笔数、交易净现金流）
- `activity_cleaned.csv`：Activity（TRADE + **REDEEM 结算**），用于**真实盈利**与按时间序回撤

**产出**：每市场的 出手次数、盈利（含结算）、回撤、买卖结构（buy_yes/no, sell_yes/no）

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# 数据路径（相对项目根或改成本机路径）
DATA_DIR = Path("/home/hliu/poly/eda/01_etl/output/0x8dxd_full")
df = pd.read_csv(DATA_DIR / "trades_cleaned.csv")
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce").astype("Int64")
df["size"] = pd.to_numeric(df["size"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["question", "side", "size", "price"])
print(f"共 {len(df)} 条成交，{df['question'].nunique()} 个市场")
df.head(3)

共 1886 条成交，425 个市场


,timestamp,datetime_utc,conditionId,question,slug,market_category,side,outcome,outcomeIndex,size,price,transactionHash,asset
0,1770965723,2026-02-13T06:55:23Z,0xc81165f2f99b17728317265e57c8709f742c395d46cc...,"Bitcoin Up or Down - February 13, 1:45AM-2:00A...",btc-updown-15m-1770965100,crypto,BUY,Up,0,1.082466,0.970,0x498b221ec7b5e2eb108ff7cbbfa3ad62a5db2f9103ec...,1147385971369581087098708231765180506003765047...
1,1770965723,2026-02-13T06:55:23Z,0x4e3214fd28507d1607c626758d3ce417a0be847e31e1...,LoL: Ground Zero Gaming vs CTBC Flying Oyster ...,lol-gz-cfo-2026-02-13,sports,SELL,CTBC Flying Oyster,1,1.490000,0.410,0x3808b6ff715f8fa644f3c22b32cf8ffcdfe9f8477a70...,8318859528513146667127761269855002022147561216...
2,1770965723,2026-02-13T06:55:23Z,0x08fa7d350c801391e7ec81d0da90f1fb05ec365f98d1...,Will the Washington Capitals win the 2026 NHL ...,will-the-washington-capitals-win-the-2026-nhl-...,other,SELL,Yes,0,11.200000,0.015,0x6d5bb4981da30b6582428b7d7cff4d36ca319a555d68...,8960753946737408461801688144376701867683750005...


In [8]:
# outcomeIndex: 0 视为 Yes 侧，1 视为 No 侧（二元市场）
df["is_yes"] = (df["outcomeIndex"].fillna(0).astype(int) == 0)
# 单笔现金流：买入为负，卖出为正
df["cashflow"] = np.where(df["side"] == "BUY", -df["size"] * df["price"], df["size"] * df["price"])

In [9]:
# 按 question 聚合：buy_yes / buy_no / sell_yes / sell_no 笔数 + 总 PnL
total_pnl = df.groupby("question")["cashflow"].sum()
idx = total_pnl.index
buy_yes  = df[(df["side"] == "BUY") & df["is_yes"]].groupby("question").size().reindex(idx).fillna(0).astype(int)
buy_no   = df[(df["side"] == "BUY") & ~df["is_yes"]].groupby("question").size().reindex(idx).fillna(0).astype(int)
sell_yes = df[(df["side"] == "SELL") & df["is_yes"]].groupby("question").size().reindex(idx).fillna(0).astype(int)
sell_no  = df[(df["side"] == "SELL") & ~df["is_yes"]].groupby("question").size().reindex(idx).fillna(0).astype(int)

agg = pd.DataFrame({
    "buy_yes": buy_yes,
    "buy_no": buy_no,
    "sell_yes": sell_yes,
    "sell_no": sell_no,
    "total_pnl": total_pnl,
}).fillna(0)
agg = agg.astype({c: int for c in ["buy_yes","buy_no","sell_yes","sell_no"]})
agg.index.name = "question"
agg.head(10)

,buy_yes,buy_no,sell_yes,sell_no,total_pnl
question,,,,,
Bakersfield Roadrunners vs. Hawaii Rainbow Warriors (W),0,0,1,0,0.600000
"Bitcoin Up or Down - February 13, 12:00AM-4:00AM ET",2,0,0,0,-19.499999
"Bitcoin Up or Down - February 13, 12AM ET",0,0,0,1,100.789110
"Bitcoin Up or Down - February 13, 1:45AM-2:00AM ET",173,38,15,56,-4533.395602
"Bitcoin Up or Down - February 13, 1:50AM-1:55AM ET",24,30,19,5,140.944426
"Bitcoin Up or Down - February 13, 1:55AM-2:00AM ET",55,54,6,0,-1826.174139
"Bitcoin Up or Down - February 13, 1AM ET",66,44,10,8,-4125.721652
"Bitcoin Up or Down - February 13, 2:00AM-2:05AM ET",0,1,0,0,-38.880000
"Bitcoin Up or Down - February 13, 2:00AM-2:15AM ET",3,5,0,0,-184.697798


In [10]:
# 按市场、按时间排序，算累计 PnL → 回撤、Sharpe
def market_drawdown_sharpe(g: pd.DataFrame) -> pd.Series:
    g = g.sort_values("timestamp")
    cf = g["cashflow"].values
    cum = np.cumsum(cf)
    peak = np.maximum.accumulate(cum)
    dd = peak - cum
    max_dd = np.max(dd) if len(dd) else 0.0
    # Sharpe: mean(cashflow)/std(cashflow)*sqrt(n)，std=0 时置 0
    n = len(cf)
    if n < 2 or np.std(cf) == 0:
        sharpe = 0.0
    else:
        sharpe = (np.mean(cf) / np.std(cf)) * np.sqrt(n)
    return pd.Series({"max_drawdown": max_dd, "sharpe": sharpe, "n_trades": n})

extra = df.groupby("question").apply(market_drawdown_sharpe)
agg = agg.join(extra)
agg.head(10)

/tmp/ipykernel_3077690/3510813473.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  extra = df.groupby("question").apply(market_drawdown_sharpe)


,buy_yes,buy_no,sell_yes,sell_no,total_pnl,max_drawdown,sharpe,n_trades
question,,,,,,,,
Bakersfield Roadrunners vs. Hawaii Rainbow Warriors (W),0,0,1,0,0.600000,0.000000,0.000000e+00,1.0
"Bitcoin Up or Down - February 13, 12:00AM-4:00AM ET",2,0,0,0,-19.499999,9.750000,-2.757716e+07,2.0
"Bitcoin Up or Down - February 13, 12AM ET",0,0,0,1,100.789110,0.000000,0.000000e+00,1.0
"Bitcoin Up or Down - February 13, 1:45AM-2:00AM ET",173,38,15,56,-4533.395602,4578.936852,-3.868667e+00,282.0
"Bitcoin Up or Down - February 13, 1:50AM-1:55AM ET",24,30,19,5,140.944426,378.360276,3.000193e-01,78.0
"Bitcoin Up or Down - February 13, 1:55AM-2:00AM ET",55,54,6,0,-1826.174139,1706.184139,-3.976001e+00,115.0
"Bitcoin Up or Down - February 13, 1AM ET",66,44,10,8,-4125.721652,4092.334852,-3.225941e+00,128.0
"Bitcoin Up or Down - February 13, 2:00AM-2:05AM ET",0,1,0,0,-38.880000,0.000000,0.000000e+00,1.0
"Bitcoin Up or Down - February 13, 2:00AM-2:15AM ET",3,5,0,0,-184.697798,182.147798,-2.035266e+00,8.0


In [11]:
# 按总 PnL 排序，看表现最好/最差市场
agg_sorted = agg.sort_values("total_pnl", ascending=False).reset_index()
agg_sorted["market_category"] = agg_sorted["question"].map(
    df.drop_duplicates("question").set_index("question")["market_category"]
)
cols = ["question", "market_category", "buy_yes", "buy_no", "sell_yes", "sell_no", "total_pnl", "max_drawdown", "sharpe", "n_trades"]
display(agg_sorted[cols].head(20))

,question,market_category,buy_yes,buy_no,sell_yes,sell_no,total_pnl,max_drawdown,sharpe,n_trades
0,LoL: Ground Zero Gaming vs CTBC Flying Oyster ...,other,0,0,8,0,5615.418960,0.000000,1.256169,8.0
1,Government shutdown on Saturday?,other,8,1,5,6,2812.171325,353.140119,1.219162,20.0
2,Will France win the 2026 FIFA World Cup?,other,0,0,1,0,1740.000000,0.000000,0.000000,1.0
3,Will the Atlanta Hawks win the 2026 NBA Finals?,sports,0,0,0,1,1460.212250,0.000000,0.000000,1.0
4,Spread: Lakers (-7.5),sports,0,0,1,0,1257.741000,0.000000,0.000000,1.0
5,Will Google have the best AI model at the end ...,other,0,0,1,0,1021.613600,0.000000,0.000000,1.0
6,Will Elon Musk post 360-379 tweets from Februa...,other,2,0,3,2,710.747781,11.195759,1.305819,7.0
7,Mavericks vs. Lakers,sports,0,0,0,3,571.098330,0.000000,2.317005,3.0
8,"Will the US next strike Iran on February 12, 2...",other,0,1,0,1,454.475162,16.999998,1.315778,2.0
9,Israel and Syria normalize relations by Decemb...,other,0,0,0,1,383.590000,0.000000,0.000000,1.0


In [12]:
# 按市场类型汇总
agg_sorted.groupby("market_category").agg(
    markets=("question", "count"),
    total_pnl=("total_pnl", "sum"),
    mean_sharpe=("sharpe", "mean"),
    mean_drawdown=("max_drawdown", "mean"),
).round(4)

,markets,total_pnl,mean_sharpe,mean_drawdown
market_category,,,,
crypto,60,-18084.5907,-459558.6135,288.3255
macro,11,65.7617,0.0000,0.0000
other,260,-1749.3042,-3.0861,16.2245
politics,55,-8808.5022,0.8839,19.8153
sports,39,2080.3703,-4.8293,36.0505


In [13]:
# 导出按市场聚合表（可选）
out_path = DATA_DIR / "markets_agg.csv"
agg_sorted[cols].to_csv(out_path, index=False, encoding="utf-8")
print(f"已写入 {out_path}")

已写入 /home/hliu/poly/eda/01_etl/output/0x8dxd_full/markets_agg.csv


---
## 基于 Activity（含 REDEEM）的按市场：真实盈利、回撤、出手次数

用 `activity_cleaned.csv` 按 **title**（市场）聚合：每笔 TRADE/REDEEM 的 usdcSize 已带符号，合计 = 真实已实现盈亏；按时间序算回撤；出手次数 = 该市场 TRADE 条数。

In [14]:
# 读 Activity（含 TRADE + REDEEM），按市场聚合
act = pd.read_csv(DATA_DIR / "activity_cleaned.csv")
act["timestamp"] = pd.to_numeric(act["timestamp"], errors="coerce")
act["usdcSize"] = pd.to_numeric(act["usdcSize"], errors="coerce")
act = act.dropna(subset=["title", "usdcSize"])
# 按市场（title）分组，计算：出手次数、盈利、回撤
def market_metrics(g: pd.DataFrame) -> pd.Series:
    g = g.sort_values("timestamp")
    cf = g["usdcSize"].values
    n_trades = int((g["type"] == "TRADE").sum())
    profit = float(np.sum(cf))
    cum = np.cumsum(cf)
    peak = np.maximum.accumulate(cum)
    max_dd = float(np.max(peak - cum)) if len(cum) else 0.0
    return pd.Series({"出手次数": n_trades, "盈利_usdc": profit, "回撤_usdc": max_dd, "总笔数": len(g)})

act_agg = act.groupby("title").apply(market_metrics, include_groups=False).reset_index()
act_agg = act_agg.rename(columns={"title": "question"})
act_agg.head(10)

,question,出手次数,盈利_usdc,回撤_usdc,总笔数
0,"Bitcoin Up or Down - February 12, 11PM ET",0.0,3284.603287,0.000000,4.0
1,"Bitcoin Up or Down - February 13, 1:00AM-1:05A...",0.0,1687.425390,0.000000,6.0
2,"Bitcoin Up or Down - February 13, 1:00AM-1:15A...",0.0,2480.039808,0.000000,4.0
3,"Bitcoin Up or Down - February 13, 1:05AM-1:10A...",0.0,2126.076381,0.000000,6.0
4,"Bitcoin Up or Down - February 13, 1:10AM-1:15A...",0.0,2761.693513,0.000000,4.0
5,"Bitcoin Up or Down - February 13, 1:15AM-1:20A...",56.0,-150.907506,658.305406,60.0
6,"Bitcoin Up or Down - February 13, 1:15AM-1:30A...",343.0,514.749851,2678.911698,345.0
7,"Bitcoin Up or Down - February 13, 1:20AM-1:25A...",76.0,548.573813,1757.779025,80.0
8,"Bitcoin Up or Down - February 13, 1:25AM-1:30A...",125.0,-821.798755,1697.838117,127.0
9,"Bitcoin Up or Down - February 13, 1:30AM-1:35A...",48.0,729.602100,1439.113850,52.0


In [18]:
# 与 trades 的 market_category 对齐，按盈利排序
act_agg["market_category"] = act_agg["question"].map(
    df.drop_duplicates("question").set_index("question")["market_category"]
)
act_sorted = act_agg.sort_values("盈利_usdc", ascending=False).reset_index(drop=True)
cols_out = ["question", "market_category", "出手次数", "盈利_usdc", "回撤_usdc", "总笔数"]
display(act_sorted[cols_out].head(10))
display(act_sorted[cols_out].tail(10))

,question,market_category,出手次数,盈利_usdc,回撤_usdc,总笔数
0,"Solana Up or Down - February 13, 12AM ET",NaN,0.0,5274.931620,0.000000,2.0
1,"Bitcoin Up or Down - February 12, 11PM ET",NaN,0.0,3284.603287,0.000000,4.0
2,"XRP Up or Down - February 13, 12AM ET",crypto,0.0,3084.906007,0.000000,2.0
3,"Bitcoin Up or Down - February 13, 1:10AM-1:15A...",NaN,0.0,2761.693513,0.000000,4.0
4,"Bitcoin Up or Down - February 13, 1:00AM-1:15A...",NaN,0.0,2480.039808,0.000000,4.0
5,"Bitcoin Up or Down - February 13, 1:05AM-1:10A...",NaN,0.0,2126.076381,0.000000,6.0
6,"Bitcoin Up or Down - February 13, 1:00AM-1:05A...",NaN,0.0,1687.425390,0.000000,6.0
7,"Bitcoin Up or Down - February 13, 1:30AM-1:45A...",NaN,146.0,1357.155495,3380.139422,148.0
8,"Ethereum Up or Down - February 13, 12AM ET",crypto,0.0,1027.801267,0.000000,2.0
9,"Bitcoin Up or Down - February 13, 2:05AM-2:10A...",NaN,47.0,925.041091,1714.898326,49.0


,question,market_category,出手次数,盈利_usdc,回撤_usdc,总笔数
35,"Ethereum Up or Down - February 13, 2:00AM-2:15...",crypto,60.0,-1429.577761,1425.894428,60.0
36,"Bitcoin Up or Down - February 13, 2:20AM-2:25A...",crypto,93.0,-1671.553702,1667.390437,93.0
37,"XRP Up or Down - February 13, 1AM ET",crypto,132.0,-1957.770159,1956.380859,132.0
38,"Bitcoin Up or Down - February 13, 2:15AM-2:20A...",NaN,90.0,-2217.593111,2214.993111,90.0
39,"Bitcoin Up or Down - February 13, 2:15AM-2:30A...",NaN,242.0,-2257.666265,2333.299292,242.0
40,"Ethereum Up or Down - February 13, 1AM ET",crypto,99.0,-2755.924447,2744.974447,99.0
41,"Solana Up or Down - February 13, 1AM ET",crypto,172.0,-3724.053806,3722.119606,172.0
42,"Bitcoin Up or Down - February 13, 2AM ET",crypto,49.0,-4463.978594,4319.498594,49.0
43,"Bitcoin Up or Down - February 13, 2:00AM-2:15A...",crypto,440.0,-5932.401370,6007.247285,440.0
44,"Bitcoin Up or Down - February 13, 1AM ET",crypto,189.0,-9518.252226,9517.203026,189.0


**说明**：`总盈利_usdc` = **TRADE（交易净现金流）** + **REDEEM（结算兑付）**。REDEEM 全量约 56k，TRADE 约 -70k，故合计约 -14k。下面按类型拆分，便于核对。

In [19]:
# TRADE vs REDEEM 拆分：总盈利 = TRADE + REDEEM（REDEEM 约 56k，TRADE 约 -70k → 合计约 -14k）
by_type = act.groupby("type")["usdcSize"].sum()
print("全量 Activity 按 type:")
print(by_type.to_string())
print(f"\n  TRADE 合计: {by_type.get('TRADE', 0):,.2f}  |  REDEEM 合计: {by_type.get('REDEEM', 0):,.2f}  |  总盈利: {by_type.sum():,.2f}")

act_cat = act.copy()
act_cat["market_category"] = act_cat["title"].map(
    df.drop_duplicates("question").set_index("question")["market_category"]
)
breakdown = act_cat.groupby(["market_category", "type"], dropna=False)["usdcSize"].sum().unstack(fill_value=0)
breakdown["总盈利_usdc"] = breakdown.sum(axis=1)
breakdown.round(2)

全量 Activity 按 type:
type
REDEEM    56188.832564
TRADE    -70021.513992

  TRADE 合计: -70,021.51  |  REDEEM 合计: 56,188.83  |  总盈利: -13,832.68


type,REDEEM,TRADE,总盈利_usdc
market_category,,,
crypto,11571.44,-40204.22,-28632.77
NaN,44617.39,-29817.30,14800.09


In [16]:
# 按市场类型汇总：市场数、总盈利、平均回撤、总出手次数
act_sorted.groupby("market_category", dropna=False).agg(
    市场数=("question", "count"),
    总盈利_usdc=("盈利_usdc", "sum"),
    平均回撤_usdc=("回撤_usdc", "mean"),
    总出手次数=("出手次数", "sum"),
).round(2)

,市场数,总盈利_usdc,平均回撤_usdc,总出手次数
market_category,,,,
crypto,17,-28632.77,2358.37,1704.0
NaN,28,14800.09,1081.23,1709.0


In [17]:
# 导出按市场：出手次数、盈利、回撤（基于 Activity，含结算）
out_act = DATA_DIR / "markets_activity_agg.csv"
act_sorted[cols_out].to_csv(out_act, index=False, encoding="utf-8")
print(f"已写入 {out_act}")

已写入 /home/hliu/poly/eda/01_etl/output/0x8dxd_full/markets_activity_agg.csv
